In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport warningswarnings.filterwarnings('ignore')import european_analyticsfrom models import mSABRfrom scipy.stats import lognormfrom matplotlib.lines import Line2D

In [ ]:
# Model parametersF0=0.05lambda_=0.04          # SABR shiftF_s=F0+lambda_beta=0.5nu=0.5                # higher than the standard-SABR notebook so MR effects are visiblerho=-0.3sigma_atm=0.01# Simulation parametersN_PATHS = 40_000SEED    = 42# Mean reversion grid: none / moderate / strongkappas = [0.0, 1.0, 4.0]labels = ['no MR (kappa=0)', 'moderate (kappa=1)', 'strong (kappa=4)']colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(kappas)))T_fixed = 2.0def build(kappa, T, n_paths=N_PATHS):    an = european_analytics.EuropeanAnalyticsMRSABR(F_s, T, beta, nu, rho, sigma_atm, lambda_, kappa)    sim = mSABR(F0=F_s, A0=an.alpha, rho=rho, T=T,                n_steps=min(max(int(500*T), 200), 5000), n_paths=n_paths,                beta=beta, nu=nu, lambda_=kappa, theta=an.alpha, seed=SEED)    return an, sim# Sanity: kappa=0 must reproduce standard SABRref = european_analytics.EuropeanAnalyticsSABR(F_s, T_fixed, beta, nu, rho, sigma_atm, lambda_)chk = european_analytics.EuropeanAnalyticsMRSABR(F_s, T_fixed, beta, nu, rho, sigma_atm, lambda_, 0.0)print(f"kappa=0 reduces to standard SABR: {np.isclose(ref.alpha, chk.alpha)}")print(f"sigma_ln_atm: {chk.sigma_ln_atm:.6f}")for kap in kappas:    an = european_analytics.EuropeanAnalyticsMRSABR(F_s, T_fixed, beta, nu, rho, sigma_atm, lambda_, kap)    print(f"kappa={kap}: nu_eff={an.nu_eff:.4f}  rho_eff={an.rho_eff:+.4f}  "          f"alpha={an.alpha:.6f}  ATM vol={an.hagan_implied_vol(F_s):.6f}")

In [ ]:
# Terminal distribution and smile vs mean reversion (T fixed)fig, axes = plt.subplots(1, 2, figsize=(14, 5))all_F_T = []print(" kappa   nu_eff  rho_eff     std       skew      kurt")for kap, lab, col in zip(kappas, labels, colors):    an, sim = build(kap, T_fixed)    F_s_paths, _ = sim.run()    F_T = F_s_paths[:, -1] - lambda_    all_F_T.append(F_T)    st = sim.compute_stats(F_s_paths - lambda_)    print(f"  {kap:.1f}    {an.nu_eff:.3f}   {an.rho_eff:+.3f}   {st['std_dev'][-1]:.5f}  "          f"{st['skewness'][-1]:+.3f}   {st['kurtosis'][-1]:+.3f}")    axes[0].hist(F_T, bins=200, density=True, histtype='stepfilled', alpha=0.35,                 color=col, edgecolor=col, linewidth=0.8, label=lab)an0 = european_analytics.EuropeanAnalyticsMRSABR(F_s, T_fixed, beta, nu, rho, sigma_atm, lambda_, 0.0)mu_ln    = np.log(F_s) - 0.5 * an0.sigma_ln_atm**2 * T_fixedsigma_ln = an0.sigma_ln_atm * np.sqrt(T_fixed)lo, hi = np.percentile(all_F_T[0], [0.3, 99.7])x_ref = np.linspace(lo, hi, 400)axes[0].plot(x_ref, lognorm.pdf(x_ref + lambda_, s=sigma_ln, scale=np.exp(mu_ln)),             'k--', lw=1.5, label='Lognormal ref')axes[0].axvline(F0, color='gray', lw=1, ls='--')axes[0].set_xlim(lo, hi)axes[0].set_xlabel(r'$F_T$')axes[0].set_ylabel('Density')axes[0].set_title(f'Terminal distribution across kappa (T = {T_fixed}y)')axes[0].legend(fontsize=8)axes[0].grid(alpha=0.3)std_sm = an0.sigma_ln_atm * F_s * np.sqrt(T_fixed)K_grid = np.linspace(F_s - 2.5*std_sm, F_s + 2.5*std_sm, 27)for kap, lab, col in zip(kappas, labels, colors):    an, sim = build(kap, T_fixed)    F_s_T = sim.run()[0][:, -1]    v_hagan = an.implied_vol_smile(K_grid)    v_mc    = an.implied_vol_smile_mc(F_s_T, K_grid)    axes[1].plot((K_grid - lambda_)*100, v_hagan*100, color=col, lw=2, label=lab)    axes[1].plot((K_grid - lambda_)*100, v_mc*100, 'o', color=col, ms=3, alpha=0.55)handles, _ = axes[1].get_legend_handles_labels()handles.append(Line2D([0],[0], marker='o', ls='', color='gray', ms=4, alpha=0.6, label='MC'))axes[1].axvline(F0*100, color='gray', lw=1, ls='--')axes[1].set_xlabel('Strike (%)')axes[1].set_ylabel('Implied vol (%)')axes[1].set_title(f'Implied vol smile across kappa (T = {T_fixed}y)')axes[1].legend(handles=handles, fontsize=8)axes[1].grid(alpha=0.3)fig.suptitle(f'Mean reversion sweep  (T = {T_fixed}y, beta = {beta}, nu = {nu}, rho = {rho})',             fontsize=13, y=1.02)plt.tight_layout()plt.show()

## Terminal Distribution and IV Smile Analysis ($\kappa$ Sweep)**Experiment:** Fix $T = 2\text{y}$, $\beta = 0.5$, $\nu = 0.5$, $\rho = -0.3$ and sweep the meanreversion speed $\kappa \in \{0, 1, 4\}$. $\kappa = 0$ recovers the standard SABR case we studiedbefore. The vol process is $dA = \kappa(\theta - A)dt + \nu A\,dW_2$ with $\theta = A_0 = \alpha$, sothe vol starts at its long-run mean and mean reversion only acts to pull it back after it wanders.**Mechanism:** The semi-analytic route maps mrSABR onto an *effective* standard SABR. The controllingquantity is the accumulated vol-of-vol variance$$w_2 = \frac{1 - e^{-2\kappa T}}{2\kappa T}, \qquad \nu_{\text{eff}} = \nu\sqrt{w_2},$$with a matching skew weight $w_1$ giving $\rho_{\text{eff}} = \rho\, w_1/\sqrt{w_2}$. Note $w_1, w_2 \to 1$as $\kappa T \to 0$, so mrSABR collapses to standard SABR. Everything below is a consequence of$\nu_{\text{eff}} < \nu$: mean reversion is, to leading order, a *vol-of-vol suppressor*.**Results:** As $\kappa$ increases the terminal distribution becomes visibly less leptokurtic — thepeak flattens and both tails thin out, moving back toward the lognormal reference. The excess kurtosisshould drop sharply from $\kappa = 0$ to $\kappa = 1$ and again to $\kappa = 4$, while the standarddeviation barely moves. The skew also weakens in magnitude, since with the vol pinned near $\theta$there is less room for $\rho$ to tilt the distribution.On the smile side, all three curves are pinned to the same ATM vol by construction ($\alpha$ iscalibrated against $\nu_{\text{eff}}$), so the differences are entirely in the wings. Higher $\kappa$gives a flatter, less curved smile — the wings collapse toward ATM. This is the direct smile-side imageof the thinner tails in the left panel: less tail mass means cheaper OTM options means lower implied volat the wings.**Takeaway:** Mean reversion acts on the smile much like *lowering* $\nu$ would. It kills curvature andtail thickness while leaving the ATM level untouched. The key difference from a plain $\nu$ reduction isthat the suppression $\sqrt{w_2}$ depends on $\kappa T$, so the same $\kappa$ has very different forceat different maturities — which is exactly what the next experiment isolates.

In [ ]:
# Maturity dependence: does mean reversion matter more at short or long T?maturities = [0.25, 0.5, 1.0, 2.0, 5.0, 10.0]kap_no, kap_mr = 0.0, 2.0kurt_no, kurt_mr, wing_no, wing_mr = [], [], [], []for T_i in maturities:    for kap, kurt_list, wing_list in [(kap_no, kurt_no, wing_no), (kap_mr, kurt_mr, wing_mr)]:        an, sim = build(kap, T_i, n_paths=40_000)        F_s_paths, _ = sim.run()        st = sim.compute_stats(F_s_paths - lambda_)        kurt_list.append(st['kurtosis'][-1])        std_i = an.sigma_ln_atm * F_s * np.sqrt(T_i)        v = an.implied_vol_smile(np.array([F_s - 2.0*std_i, F_s]))        wing_list.append((v[0] - v[1]) * 100)fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].plot(maturities, np.maximum(kurt_no, 1e-2), 'o-', color='#993C1D', lw=2, label='no MR (kappa=0)')axes[0].plot(maturities, np.maximum(kurt_mr, 1e-2), 's-', color='#0F6E56', lw=2, label='MR (kappa=2)')axes[0].set_yscale('log')axes[0].set_xlabel('T (years)')axes[0].set_ylabel('Excess kurtosis of $F_T$ (log scale)')axes[0].set_title('Tail fatness vs maturity')axes[0].legend(fontsize=8)axes[0].grid(alpha=0.3, which='both')axes[1].plot(maturities, wing_no, 'o-', color='#993C1D', lw=2, label='no MR (kappa=0)')axes[1].plot(maturities, wing_mr, 's-', color='#0F6E56', lw=2, label='MR (kappa=2)')axes[1].set_xlabel('T (years)')axes[1].set_ylabel('Left-wing lift over ATM (vol pts)')axes[1].set_title('Smile steepness vs maturity')axes[1].legend(fontsize=8)axes[1].grid(alpha=0.3)fig.suptitle(f'Maturity dependence of mean reversion  (beta = {beta}, nu = {nu}, rho = {rho})',             fontsize=13, y=1.02)plt.tight_layout()plt.show()print("   T     kurt(noMR)   kurt(MR)    wing(noMR)  wing(MR)   sqrt(w2)")for i, T_i in enumerate(maturities):    _, w2 = european_analytics.mr_weights(kap_mr, T_i)    print(f"  {T_i:5.2f}   {kurt_no[i]:+10.3f}  {kurt_mr[i]:+8.3f}    {wing_no[i]:7.2f}   {wing_mr[i]:6.2f}    {np.sqrt(w2):.3f}")

## Maturity Dependence of Mean Reversion**Experiment:** Fix $\beta = 0.5$, $\nu = 0.5$, $\rho = -0.3$ and compare $\kappa = 0$ against$\kappa = 2$ while sweeping $T$ across $[0.25, 0.5, 1, 2, 5, 10]$ years. Left panel tracks the excesskurtosis of the simulated $F_T$; right panel tracks the left-wing lift (implied vol at $F_s - 2\sigma$minus ATM vol) from the effective-Hagan smile.**Results:** At short maturities ($T = 0.25$, $0.5$) the two curves nearly coincide — mean reversion hashad almost no time to act, so mrSABR and standard SABR are effectively the same model. The gap thenwidens monotonically with $T$. Without mean reversion the kurtosis grows explosively at long maturities:the vol process $A_t$ is a driftless geometric Brownian motion, so $\mathbb{E}[A_t^2]$ grows like$e^{\nu^2 t}$ without bound, and by $T = 10\text{y}$ a handful of paths carry enormous vol and the samplekurtosis blows up (this is why the left panel needs a log axis, and why sample kurtosis at $T = 10$ isitself unstable across seeds). With $\kappa = 2$ the kurtosis instead flattens out to a roughly constantlevel — the vol process reaches a stationary regime and stops accumulating variance.The right panel shows the same story in smile terms. The no-MR wing keeps steepening roughly like$\nu\sqrt{T}$, while the mean-reverting wing grows much more slowly and starts to saturate, because$\nu_{\text{eff}}\sqrt{T} = \nu\sqrt{w_2 T} \to \nu/\sqrt{2\kappa}$ as $\kappa T \to \infty$ — a finitelimit rather than unbounded growth.**Takeaway:** Mean reversion matters **far more at long maturities**, and the reason is that the wholeeffect is governed by the product $\kappa T$, not $\kappa$ alone. The suppression factor$\sqrt{w_2} \approx 1$ when $\kappa T \ll 1$ and decays like $1/\sqrt{2\kappa T}$ when $\kappa T \gg 1$.This is precisely the pathology mrSABR was introduced to fix: standard SABR's unbounded vol varianceproduces unrealistic long-dated smiles, and mean reversion caps it.

In [ ]:
# ATM vs wings: where does mean reversion bite? (long maturity)T_long = 5.0an_ref = european_analytics.EuropeanAnalyticsMRSABR(F_s, T_long, beta, nu, rho, sigma_atm, lambda_, 0.0)std_l  = an_ref.sigma_ln_atm * F_s * np.sqrt(T_long)K_grid_l = np.linspace(F_s - 2.5*std_l, F_s + 2.5*std_l, 31)fig, axes = plt.subplots(1, 2, figsize=(14, 5))smiles = {}for kap, lab, col in zip(kappas, labels, colors):    an, sim = build(kap, T_long)    F_s_T = sim.run()[0][:, -1]    v_hagan = an.implied_vol_smile(K_grid_l)    v_mc    = an.implied_vol_smile_mc(F_s_T, K_grid_l)    smiles[kap] = v_hagan    axes[0].plot((K_grid_l - lambda_)*100, v_hagan*100, color=col, lw=2, label=lab)    axes[0].plot((K_grid_l - lambda_)*100, v_mc*100, 'o', color=col, ms=3, alpha=0.55)handles, _ = axes[0].get_legend_handles_labels()handles.append(Line2D([0],[0], marker='o', ls='', color='gray', ms=4, alpha=0.6, label='MC'))axes[0].axvline(F0*100, color='gray', lw=1, ls='--')axes[0].set_xlabel('Strike (%)')axes[0].set_ylabel('Implied vol (%)')axes[0].set_title(f'Smile across kappa (T = {T_long}y)')axes[0].legend(handles=handles, fontsize=8)axes[0].grid(alpha=0.3)# vol change vs no-MR baseline, by moneynessbase = smiles[0.0]moneyness = (K_grid_l - F_s) / std_lfor kap, lab, col in zip(kappas[1:], labels[1:], colors[1:]):    axes[1].plot(moneyness, (smiles[kap] - base)*100, color=col, lw=2, label=lab)axes[1].axhline(0, color='gray', lw=1, ls='--')axes[1].axvline(0, color='gray', lw=1, ls='--')axes[1].set_xlabel('Moneyness (std devs from ATM)')axes[1].set_ylabel('Implied vol change vs no-MR (vol pts)')axes[1].set_title(f'Where mean reversion bites (T = {T_long}y)')axes[1].legend(fontsize=8)axes[1].grid(alpha=0.3)fig.suptitle(f'ATM vs wings  (T = {T_long}y, beta = {beta}, nu = {nu}, rho = {rho})',             fontsize=13, y=1.02)plt.tight_layout()plt.show()atm_idx = np.argmin(np.abs(K_grid_l - F_s))print(f"ATM vol change vs no-MR:  kappa=1: {(smiles[1.0][atm_idx]-base[atm_idx])*100:+.3f} pts   "      f"kappa=4: {(smiles[4.0][atm_idx]-base[atm_idx])*100:+.3f} pts")print(f"-2.5 std wing change:     kappa=1: {(smiles[1.0][0]-base[0])*100:+.3f} pts   "      f"kappa=4: {(smiles[4.0][0]-base[0])*100:+.3f} pts")print(f"+2.5 std wing change:     kappa=1: {(smiles[1.0][-1]-base[-1])*100:+.3f} pts   "      f"kappa=4: {(smiles[4.0][-1]-base[-1])*100:+.3f} pts")

## ATM vs Wings: Where Does Mean Reversion Matter?**Experiment:** Fix $T = 5\text{y}$ (long enough that $\kappa T$ is large and mean reversion has realforce), $\beta = 0.5$, $\nu = 0.5$, $\rho = -0.3$, and compare the full smile across$\kappa \in \{0, 1, 4\}$. The right panel plots the *change* in implied vol relative to the no-MRbaseline as a function of moneyness in standard deviations, which isolates where the effect lives.**Results:** The ATM point is essentially unchanged across all $\kappa$ — by construction, since$\alpha$ is recalibrated against $\nu_{\text{eff}}$ so every curve reproduces the same $\sigma_{atm}$.The change is near zero at moneyness 0 and grows in magnitude as we move out in either direction,producing a characteristic U-shape (inverted) in the right panel. Deep OTM/ITM strikes lose the mostimplied vol, and the effect is asymmetric: with $\rho < 0$ the left wing was the steepest to begin with,so it also has the most to lose.The reason is structural. From Hagan's expansion, the ATM level is driven by $\alpha/f^{1-\beta}$ with onlya small $O(\nu^2 T)$ correction, whereas the wings are driven by the $z/x(z)$ factor whose curvature isfirst-order in $\nu$. Since mean reversion enters (to leading order) purely as $\nu \to \nu_{\text{eff}}$,it barely touches the ATM level but directly attacks the wings.**Takeaway:** Mean reversion is a **wing/tail parameter, not a level parameter**. This makes itcomplementary to the parameters from the standard SABR study: $\sigma_{atm}$ moves the level, $\rho$ setsthe skew direction, $\nu$ sets the curvature, and $\kappa$ discounts that curvature by an amount thatdepends on how far out the expiry is. For pricing, this means mean reversion is nearly irrelevant for ATMoptions and matters most for long-dated deep OTM/ITM strikes — exactly the corner where long-datedstandard SABR smiles are least believable.

In [ ]:
# Joint (kappa, T) impact heatmap: % reduction in left-wing lift vs no-MR at same Tkappa_grid = np.array([0.0, 0.5, 1.0, 2.0, 4.0, 8.0])T_grid     = np.array([0.25, 0.5, 1.0, 2.0, 5.0, 10.0])impact = np.zeros((len(kappa_grid), len(T_grid)))for j, T_j in enumerate(T_grid):    an_b = european_analytics.EuropeanAnalyticsMRSABR(F_s, T_j, beta, nu, rho, sigma_atm, lambda_, 0.0)    std_j = an_b.sigma_ln_atm * F_s * np.sqrt(T_j)    K_wing = np.array([F_s - 2.0*std_j, F_s])    v_b = an_b.implied_vol_smile(K_wing)    base_lift = v_b[0] - v_b[1]    for i, kap in enumerate(kappa_grid):        an_ij = european_analytics.EuropeanAnalyticsMRSABR(F_s, T_j, beta, nu, rho, sigma_atm, lambda_, kap)        v = an_ij.implied_vol_smile(K_wing)        impact[i, j] = (1 - (v[0]-v[1])/base_lift)*100 if base_lift > 1e-9 else 0.0fig, ax = plt.subplots(figsize=(9, 6))im = ax.imshow(impact, origin='lower', aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)ax.set_xticks(range(len(T_grid)))ax.set_xticklabels([str(t) for t in T_grid])ax.set_yticks(range(len(kappa_grid)))ax.set_yticklabels([str(k) for k in kappa_grid])ax.set_xlabel('T (years)')ax.set_ylabel('kappa (mean reversion speed)')for i in range(len(kappa_grid)):    for j in range(len(T_grid)):        val = impact[i, j]        txt_col = 'white' if val > 60 else 'black'        ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=9, color=txt_col)plt.colorbar(im, ax=ax, label='% reduction in smile wing vs no-MR (same T)')ax.set_title(f'Mean reversion impact on smile steepness  '             f'(beta = {beta}, nu = {nu}, rho = {rho})', fontsize=12)plt.tight_layout()plt.show()print("  nu_eff/nu = sqrt(w2) at each (kappa, T):")print("  kappa\\T " + "  ".join(f"{t:6.2f}" for t in T_grid))for kap in kappa_grid:    row = [np.sqrt(european_analytics.mr_weights(kap, T_j)[1]) for T_j in T_grid]    print(f"  {kap:5.1f}  " + "  ".join(f"{r:6.3f}" for r in row))

## Joint ($\kappa$, $T$) Impact Heatmap**Experiment:** Fix $\beta = 0.5$, $\nu = 0.5$, $\rho = -0.3$, then jointly sweep $\kappa$ across$[0, 0.5, 1, 2, 4, 8]$ and $T$ across $[0.25, 0.5, 1, 2, 5, 10]$ years. At each cell we compute thepercentage reduction in the left-wing lift (implied vol at $-2\sigma$ minus ATM vol) relative to theno-MR smile at the same maturity. This is the semi-analytic mirror of the $(\nu, T)$ divergence heatmapfrom the standard SABR study, and needs no MC since it only uses the effective-Hagan smile.**Results:** The heatmap is organised along **diagonals of constant $\kappa T$**, not along either axisalone. The bottom row ($\kappa = 0$) is identically zero by construction. The bottom-left region (small$\kappa$, short $T$) stays near zero — mean reversion is present but has no time to act, so the smile isindistinguishable from standard SABR. The top-right corner (large $\kappa$, long $T$) shows the largestreductions, where mean reversion removes most of the smile curvature that standard SABR would produce.Crucially, a large $\kappa$ at $T = 0.25$ and a small $\kappa$ at $T = 10$ can land in the same colourband, because both correspond to a similar $\kappa T$.The printed $\sqrt{w_2}$ table makes the mechanism explicit: it is a pure function of $\kappa T$, fallingfrom $1$ toward $1/\sqrt{2\kappa T}$. For example $\kappa = 8, T = 0.25$ and $\kappa = 2, T = 1$ share$\kappa T = 2$ and therefore share the same suppression.**Takeaway:** $\kappa T$ is the single dimensionless group that controls the entire mean-reversion effectin this effective-parameter approximation, in the same way $\nu^2 T$ controlled the accuracy of Hagan'sexpansion in the standard SABR study. Practically, mean reversion is safely ignorable for short-datedproducts at any plausible $\kappa$, but it becomes a first-order modelling choice for long-datedderivatives — which is exactly the regime the project brief identifies as standard SABR's weak point.

## Caveats and Next Steps**On the semi-analytic formula.** The effective-parameter mapping used here($\nu_{\text{eff}} = \nu\sqrt{w_2}$, $\rho_{\text{eff}} = \rho w_1/\sqrt{w_2}$) is derived in the spirit ofthe effective-medium argument in Appendix C of Hagan et al. (2002), where time-dependent vol parameters arereplaced by time-averaged effective constants. It has the right limits ($\kappa \to 0$ recovers standardSABR exactly, verified in the setup cell) and the right qualitative behaviour, but it is **not** atranscription of Hagan's (2020) mrSABR coefficients. Before this goes in the report, `mr_weights` should bereplaced with the exact asymptotic expressions from that paper; the class structure and every experimentabove will carry over unchanged, since only that one function needs swapping.**MC is the ground truth.** The `mSABR` simulation makes no approximation beyond Euler discretisation, sothe Hagan-vs-MC gaps in the smile plots are a genuine measure of where the effective-SABR approximationbreaks down — the same diagnostic used for standard SABR. Expect the gap to widen at large $\nu$ andlarge $T$, and note that a single effective SABR cannot in principle match every moment of the mrSABRterminal distribution.**Next step (moment matching).** The project brief asks for a second, independent route to effectiveparameters: matching variance, skewness and kurtosis of the simulated $X_T = F_T - F_0$ against standardSABR (fixing $\beta = 0$ without loss of generality) and solving for $(\alpha_{\text{eff}},\nu_{\text{eff}}, \rho_{\text{eff}})$. Since `compute_stats` already returns exactly those moments, thatcomparison slots directly onto the machinery here — and the interesting question becomes whether the tworoutes agree at leading order and where they diverge as $\kappa T$ and $\nu^2 T$ grow.